### Pydantic

In [1]:

import os 
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ["Google_API_KEY"]=os.getenv("Google_API_KEY")

In [2]:
model=init_chat_model(model="google_genai:gemini-2.5-flash")

In [ ]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(...,description="Title of the movie")
    hero:str=Field(...,description="Name of the Main Actor in Movie")
    year:int=Field(...,description="In which year movie get released")
    rating:float=Field(...,description="Rating of the movie")
    revenue:float=Field(...,description="How much revenue is gain by the movie in Thousand crore in INR")


In [ ]:
structured_output=model.with_structured_output(Movie)
response=structured_output.invoke("about Bahubali Movie")
print(response)

title='Baahubali 2: The Conclusion' hero='Prabhas' year=2017 rating=8.3 revenue=1.8


In [4]:
from typing_extensions import TypedDict, Annotated

class Movie(TypedDict):
    title:Annotated[str,...,"The Title of the Movie"]
    year:Annotated[int,...,'in which year it release']
    rating:Annotated[float,...,'how much rating it get']
    
type_dict_model=model.with_structured_output(Movie)
response=type_dict_model.invoke("about Avengers movie")
response

{'title': 'The Avengers', 'year': 2012, 'rating': 8.0}

In [5]:
from langchain.agents import create_agent
from pydantic import BaseModel,Field

class weather(BaseModel):
    location:str=Field(...,description='it shows the location')
    tempreture:float=Field(...,description='it represent the tempretuture in degree celcious')
    



def getweather(location:str)->str:
    '''this function return weather'''
    return f'the weather is sunny and tempreture is 45 degree in nandgaon peth'
    
    
agent=create_agent(
    model='google_genai:gemini-2.5-flash-lite',
    tools=[getweather],
    system_prompt='you are polite asistent',
    response_format=weather
)
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "give the temperature and weather of nandgaon peth"
        }
    ]
})
result

{'messages': [HumanMessage(content='give the temperature and weather of nandgaon peth', additional_kwargs={}, response_metadata={}, id='e9840ccf-dad6-44a6-ba46-1fee6d9fe4be'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'getweather', 'arguments': '{"location": "nandgaon peth"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e6049-46ae-7e50-afba-946dc8eb0437-0', tool_calls=[{'name': 'getweather', 'args': {'location': 'nandgaon peth'}, 'id': '5aff175e-db8d-48db-b6ef-0c691fa795e8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 108, 'output_tokens': 18, 'total_tokens': 126, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='the weather is sunny and tempreture is 45 degree in nandgaon peth', name='getweather', id='b3f18652-da22-4e60-b513-222357ad511b', tool_call_id='5aff175e-db8d-48db-b6ef-0c691fa795e8'),
  AI

In [7]:
result['structured_response']

weather(location='nandgaon peth', tempreture=45.0)